# Spectral Compressed Sensing: Numerical Simulations

This notebook demonstrates the Projected Gradient Descent (PGD) algorithm for Spectral Compressed Sensing, as described in:

> Cai, J.-F., Wang, T., & Wei, K. (2018). *Spectral Compressed Sensing via Projected Gradient Descent*. SIAM Journal on Optimization, 28(3), 2625-2653.

Run each cell sequentially to see the algorithm in action.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng

# Import our modules
from utils import (
    generate_signal, build_hankel, hankel_adj,
    sample_with_replacement, apply_sampling_operator,
    relative_error, recovery_success, build_preconditioner
)
from pgd_algorithm import pgd_recovery, initialize_pgd
from fiht_baseline import fiht_recovery

# For reproducibility
rng = default_rng(42)

print("Modules loaded successfully!")

## 2. Generate a Spectrally Sparse Signal

A spectrally sparse signal is a superposition of $r$ complex sinusoids:
\[x(t) = \sum_{k=1}^{r} d_k e^{(2\pi i f_k - \tau_k)t}\]

where $f_k$ are frequencies, $d_k$ are complex amplitudes, and $\tau_k$ are damping factors.

In [ ]:
# Parameters
n = 128    # Signal length
r = 5      # Number of sinusoids

# Generate signal
x_true, params = generate_signal(n, r, damped=False, rng=rng)

print("Signal parameters:")
print(f"  Length n = {n}")
print(f"  Rank r = {r}")
print(f"  Frequencies: {np.round(params['frequencies'], 4)}")
print(f"  Damping: {params['damping']}")
print(f"  Signal energy: {np.linalg.norm(x_true)**2:.4f}")

# Visualize the signal
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
t = np.arange(n)

# Real part
axes[0, 0].plot(t, x_true.real)
axes[0, 0].set_title('Real Part')
axes[0, 0].set_xlabel('Time')

# Imaginary part
axes[0, 1].plot(t, x_true.imag)
axes[0, 1].set_title('Imaginary Part')
axes[0, 1].set_xlabel('Time')

# Magnitude
axes[1, 0].plot(t, np.abs(x_true))
axes[1, 0].set_title('Magnitude')
axes[1, 0].set_xlabel('Time')

# Hankel matrix
H = build_hankel(x_true)
im = axes[1, 1].imshow(np.abs(H), cmap='hot', aspect='auto')
axes[1, 1].set_title(f'Hankel Matrix (rank={np.linalg.matrix_rank(H, tol=1e-6)})')
axes[1, 1].set_xlabel('Column')
axes[1, 1].set_ylabel('Row')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print(f"\nHankel matrix shape: {H.shape}")
print(f"Hankel matrix rank: {np.linalg.matrix_rank(H, tol=1e-6)}")

## 3. Sample the Signal

We observe only a subset of entries (50% in this case) and attempt recovery.

In [ ]:
# Sampling
m_ratio = 0.5  # 50% sampling
m = int(m_ratio * n)

Omega = sample_with_replacement(n, m, rng=rng)
y = apply_sampling_operator(x_true, Omega)

print(f"Sampling: {m}/{n} = {m_ratio*100:.0f}% entries observed")

# Visualize sampling
mask = np.zeros(n)
mask[np.unique(Omega)] = 1

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(t, x_true.real, 'b-', alpha=0.3, label='True signal')
axes[0].scatter(t[Omega], x_true[Omega].real, c='red', s=10, label='Samples')
axes[0].set_title('Real Part with Samples')
axes[0].set_xlabel('Time')
axes[0].legend()

axes[1].stem(t, mask, basefmt=' ')
axes[1].set_title('Sampling Pattern')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('Sampled (1=yes, 0=no)')

plt.tight_layout()
plt.show()

## 4. Run PGD Recovery

In [ ]:
# Run PGD
print("Running PGD recovery...")

result_pgd = pgd_recovery(
    y, Omega, r, n,
    max_iter=300,
    tol=1e-6,
    x_true=x_true,       # For monitoring convergence
    verbose=True,         # Print progress
    return_history=True   # Return convergence history
)

print(f"\n=== PGD Results ===")
print(f"Converged: {result_pgd['converged']}")
print(f"Iterations: {result_pgd['iterations']}")
print(f"Final relative error: {result_pgd['rel_error']:.2e}")
print(f"Runtime: {result_pgd['time']:.3f}s")

## 5. Run FIHT Recovery (Baseline)

In [ ]:
# Run FIHT
print("Running FIHT recovery...")

result_fiht = fiht_recovery(
    y, Omega, r, n,
    max_iter=300,
    tol=1e-6,
    x_true=x_true,
    verbose=True,
    return_history=True
)

print(f"\n=== FIHT Results ===")
print(f"Converged: {result_fiht['converged']}")
print(f"Iterations: {result_fiht['iterations']}")
print(f"Final relative error: {result_fiht['rel_error']:.2e}")
print(f"Runtime: {result_fiht['time']:.3f}s")

## 6. Visualize Recovery Results

In [ ]:
x_pgd = result_pgd['x']
x_fiht = result_fiht['x']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# True signal
axes[0, 0].plot(t, x_true.real, 'b-', linewidth=1)
axes[0, 0].set_title('True Signal (Real Part)')
axes[0, 0].set_xlabel('Time')

# PGD recovery
axes[0, 1].plot(t, x_pgd.real, 'g-', linewidth=1)
axes[0, 1].set_title(f'PGD Recovery (err={result_pgd["rel_error"]:.2e})')
axes[0, 1].set_xlabel('Time')

# FIHT recovery
axes[0, 2].plot(t, x_fiht.real, 'r-', linewidth=1)
axes[0, 2].set_title(f'FIHT Recovery (err={result_fiht["rel_error"]:.2e})')
axes[0, 2].set_xlabel('Time')

# Recovery error (PGD)
axes[1, 0].plot(t, np.abs(x_pgd - x_true), 'g-', linewidth=1)
axes[1, 0].set_title('PGD Absolute Error')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('|Error|')
axes[1, 0].set_yscale('log')

# Recovery error (FIHT)
axes[1, 1].plot(t, np.abs(x_fiht - x_true), 'r-', linewidth=1)
axes[1, 1].set_title('FIHT Absolute Error')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('|Error|')
axes[1, 1].set_yscale('log')

# Overlay comparison
axes[1, 2].plot(t, x_true.real, 'b-', linewidth=2, label='True', alpha=0.7)
axes[1, 2].plot(t, x_pgd.real, 'g--', linewidth=1.5, label='PGD')
axes[1, 2].plot(t, x_fiht.real, 'r:', linewidth=1.5, label='FIHT')
axes[1, 2].set_title('Overlay Comparison')
axes[1, 2].set_xlabel('Time')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 7. Convergence Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Relative error vs iteration
pgd_history = result_pgd['history']['rel_errors']
fiht_history = result_fiht['history']['rel_errors']

axes[0].semilogy(range(len(pgd_history)), pgd_history, 'b-', label='PGD', linewidth=2)
axes[0].semilogy(range(len(fiht_history)), fiht_history, 'r-', label='FIHT', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Relative Error')
axes[0].set_title('Convergence Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Objective value vs iteration
pgd_obj = result_pgd['history']['obj_values']
fiht_obj = result_fiht['history']['obj_values']

axes[1].semilogy(range(len(pgd_obj)), pgd_obj, 'b-', label='PGD', linewidth=2)
axes[1].semilogy(range(len(fiht_obj)), fiht_obj, 'r-', label='FIHT', linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Objective Value')
axes[1].set_title('Objective Function')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PGD converged in {len(pgd_history)} iterations")
print(f"FIHT converged in {len(fiht_history)} iterations")

## 8. Test Different Sampling Ratios

See how recovery performance varies with the amount of observed data.

In [ ]:
m_ratios = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
num_trials = 5

pgd_errors = []
fiht_errors = []

for m_ratio in m_ratios:
    m = int(m_ratio * n)
    pgd_errs = []
    fiht_errs = []
    
    for _ in range(num_trials):
        x, _ = generate_signal(n, r, damped=False, rng=rng)
        Omega = sample_with_replacement(n, m, rng=rng)
        y = apply_sampling_operator(x, Omega)
        
        res_pgd = pgd_recovery(y, Omega, r, n, max_iter=200, tol=1e-3,
                               x_true=x, verbose=False)
        pgd_errs.append(res_pgd['rel_error'])
        
        res_fiht = fiht_recovery(y, Omega, r, n, max_iter=200, tol=1e-3,
                                 x_true=x, verbose=False)
        fiht_errs.append(res_fiht['rel_error'])
    
    pgd_errors.append(pgd_errs)
    fiht_errors.append(fiht_errs)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

pgd_mean = [np.mean(e) for e in pgd_errors]
pgd_std = [np.std(e) for e in pgd_errors]
fiht_mean = [np.mean(e) for e in fiht_errors]
fiht_std = [np.std(e) for e in fiht_errors]

ax.errorbar(m_ratios, pgd_mean, yerr=pgd_std, marker='o', label='PGD',
            capsize=5, linewidth=2)
ax.errorbar(m_ratios, fiht_mean, yerr=fiht_std, marker='s', label='FIHT',
            capsize=5, linewidth=2)

ax.set_xlabel('Sampling Ratio m/n')
ax.set_ylabel('Relative Error')
ax.set_title(f'Recovery vs Sampling Ratio (n={n}, r={r})')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Run Full Experiments

Execute the complete experiment suite to generate all results.

**Note:** This will take some time. For a quick test, use `quick=True`.

In [ ]:
from experiments import (
    run_phase_transition, plot_phase_transition,
    run_convergence_analysis, plot_convergence,
    run_noise_robustness, plot_noise_robustness,
    save_results
)

# Set to True for quick test runs
quick_mode = True

if quick_mode:
    print("Running in QUICK mode (reduced parameters)\n")
    num_trials_pt = 3
    r_values_pt = np.arange(2, 8, 2)
    m_ratios_pt = np.linspace(0.3, 1.0, 8)
else:
    print("Running FULL experiments\n")
    num_trials_pt = 10
    r_values_pt = np.arange(2, 16, 2)
    m_ratios_pt = np.linspace(0.1, 1.0, 19)

# Phase Transition
print("=== Phase Transition (PGD) ===")
results_pt = run_phase_transition(
    n=128,
    r_values=r_values_pt,
    m_ratios=m_ratios_pt,
    num_trials=num_trials_pt,
    damped=False,
    algorithm='pgd',
    rng=rng,
    verbose=True
)

fig = plot_phase_transition(results_pt)
plt.show()

## Summary

This notebook demonstrated:
1. **Signal generation** - Creating spectrally sparse signals
2. **Sampling** - Random observation of partial entries
3. **PGD recovery** - Projected gradient descent reconstruction
4. **FIHT baseline** - Comparison with fast iterative hard thresholding
5. **Visualization** - Signal plots, convergence curves, error analysis

For batch execution of all experiments, use:
```bash
python run_all_experiments.py --output-dir results
```